In [1]:
# =========================================================
# FEATURE ENGINEERING
# =========================================================

# -------------------------------
# 1. Install Required Library
# -------------------------------
!pip install textblob -q

# -------------------------------
# 2. Import Libraries
# -------------------------------
import pandas as pd
import numpy as np
import scipy.sparse as sp

from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer

# -------------------------------
# 3. Upload Dataset
# -------------------------------
from google.colab import files
uploaded = files.upload()

# Load preprocessed dataset
df = pd.read_csv('/content/preprocessed_zomato_data.csv')

# -------------------------------
# 4. Display Dataset
# -------------------------------
print(df.head())

# -------------------------------
# 5. Create Numerical Features
# -------------------------------

# Tweet Length
df['tweet_length'] = df['cleaned_tweet'].astype(str).apply(len)

# Word Count
df['word_count'] = df['cleaned_tweet'].astype(str).apply(
    lambda x: len(x.split())
)

# Engagement Score

if 'likes' in df.columns and 'retweets' in df.columns:
    df['engagement_score'] = df['likes'] + df['retweets']
else:
    df['engagement_score'] = 0

# -------------------------------
# 6. TF-IDF Vectorization
# -------------------------------

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9
)

X_text = tfidf.fit_transform(df['cleaned_tweet'].astype(str))

# -------------------------------
# 7. Combine Features
# -------------------------------

num_features = df[['tweet_length',
                   'word_count',
                   'engagement_score']].values

X = sp.hstack((X_text, num_features))

# -------------------------------
# 8. Create Sentiment Labels
# -------------------------------

def get_sentiment(text):

    polarity = TextBlob(text).sentiment.polarity

    if polarity > 0:
        return 1

    elif polarity < 0:
        return -1

    else:
        return 0

df['sentiment'] = df['cleaned_tweet'].astype(str).apply(get_sentiment)

y = df['sentiment']

# -------------------------------
# 9. Output Shapes
# -------------------------------

print("Feature Matrix Shape:", X.shape)
print("Labels Shape:", y.shape)

print("\nFeature Engineering Completed Successfully")

Saving preprocessed_zomato_data.csv to preprocessed_zomato_data (1).csv
                    id                 created_at        date      time  \
0  1415522329195515904  2021-07-15 04:04:31+00:00  2021-07-15  04:04:31   
1  1415522472628162564  2021-07-15 04:03:51+00:00  2021-07-15  04:03:51   
2  1415522463052537860  2021-07-15 04:03:48+00:00  2021-07-15  04:03:48   
3  1415522282039136256  2021-07-15 04:03:05+00:00  2021-07-15  04:03:05   
4  1415522222912020480  2021-07-15 04:02:51+00:00  2021-07-15  04:02:51   

   timezone              user_id         username  \
0         0  1343225527813898240  thakura47791042   
1         0  1312265407047163904      tripurateer   
2         0  1321156716990353409    ankitkakkad11   
3         0             99065681       schandra78   
4         0           1693990502           silelf   

                           name  \
0                    Aman Singh   
1                   TRIPURATEER   
2                  Ankit Kakkad   
3               Sa

In [4]:
# =========================================================
# MODEL BUILDING
# =========================================================

# -------------------------------
# 1. Import Libraries
# -------------------------------

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

# -------------------------------
# 2. Train-Test Split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# -------------------------------
# 3. Logistic Regression Model
# -------------------------------

# Changed solver and increased iterations
# to avoid convergence warning

lr_model = LogisticRegression(
    max_iter=2000,
    solver='liblinear'
)

lr_model.fit(X_train, y_train)

print("Logistic Regression Model Trained Successfully")

# -------------------------------
# 4. Support Vector Machine
# -------------------------------

svm_model = LinearSVC()

svm_model.fit(X_train, y_train)

print("SVM Model Trained Successfully")

# -------------------------------
# 5. Naive Bayes Model
# -------------------------------

nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

print("Naive Bayes Model Trained Successfully")

# -------------------------------
# 6. Sample Prediction
# -------------------------------

import scipy.sparse as sp
import numpy as np

sample_text = ["zomato ipo is performing very well"]

sample_tfidf = tfidf.transform(sample_text)

sample_num = np.array([
    [
        len(sample_text[0]),
        len(sample_text[0].split()),
        0
    ]
])

sample_final = sp.hstack((sample_tfidf, sample_num))

prediction = svm_model.predict(sample_final)

if prediction[0] == 1:
    print("Predicted Sentiment: Positive")

elif prediction[0] == -1:
    print("Predicted Sentiment: Negative")

else:
    print("Predicted Sentiment: Neutral")

print("\nModel Building Completed Successfully")

Logistic Regression Model Trained Successfully
SVM Model Trained Successfully
Naive Bayes Model Trained Successfully
Predicted Sentiment: Neutral

Model Building Completed Successfully
